# Fusion Retrieval in Document Search

## Overview

This code implements a Fusion Retrieval system that combines vector-based similarity search with keyword-based BM25 retrieval. The approach aims to leverage the strengths of both methods to improve the overall quality and relevance of document retrieval.

## Motivation

Traditional retrieval methods often rely on either semantic understanding (vector-based) or keyword matching (BM25). Each approach has its strengths and weaknesses. Fusion retrieval aims to combine these methods to create a more robust and accurate retrieval system that can handle a wider range of queries effectively.

## Key Components

1. PDF processing and text chunking
2. Vector store creation using FAISS and OpenAI embeddings
3. BM25 index creation for keyword-based retrieval
4. Custom fusion retrieval function that combines both methods

## Method Details

### Document Preprocessing

1. The PDF is loaded and split into chunks using RecursiveCharacterTextSplitter.
2. Chunks are cleaned by replacing 't' with spaces (likely addressing a specific formatting issue).

### Vector Store Creation

1. OpenAI embeddings are used to create vector representations of the text chunks.
2. A FAISS vector store is created from these embeddings for efficient similarity search.

### BM25 Index Creation

1. A BM25 index is created from the same text chunks used for the vector store.
2. This allows for keyword-based retrieval alongside the vector-based method.

### Fusion Retrieval Function

The `fusion_retrieval` function is the core of this implementation:

1. It takes a query and performs both vector-based and BM25-based retrieval.
2. Scores from both methods are normalized to a common scale.
3. A weighted combination of these scores is computed (controlled by the `alpha` parameter).
4. Documents are ranked based on the combined scores, and the top-k results are returned.

## Benefits of this Approach

1. Improved Retrieval Quality: By combining semantic and keyword-based search, the system can capture both conceptual similarity and exact keyword matches.
2. Flexibility: The `alpha` parameter allows for adjusting the balance between vector and keyword search based on specific use cases or query types.
3. Robustness: The combined approach can handle a wider range of queries effectively, mitigating weaknesses of individual methods.
4. Customizability: The system can be easily adapted to use different vector stores or keyword-based retrieval methods.

## Conclusion

Fusion retrieval represents a powerful approach to document search that combines the strengths of semantic understanding and keyword matching. By leveraging both vector-based and BM25 retrieval methods, it offers a more comprehensive and flexible solution for information retrieval tasks. This approach has potential applications in various fields where both conceptual similarity and keyword relevance are important, such as academic research, legal document search, or general-purpose search engines.

# Package Installation and Imports

The cell below installs all necessary packages required to run this notebook.


In [1]:
# Install required packages
#!uv pip install langchain langchain-openai langchain-community langchain-text-splitters faiss-cpu numpy python-dotenv rank-bm25 pymupdf

In [2]:
#!uv pip install numpy rank-bm25 pymupdf deepeval

In [3]:
import os
import sys
from dotenv import load_dotenv
from langchain_community.docstore.document import Document

from typing import List
from rank_bm25 import BM25Okapi
import numpy as np

from langchain_openai import AzureOpenAIEmbeddings


from utils.helper_functions import *
# from utils.evaluate_rag import *

# Load environment variables from a .env file
load_dotenv(override=True)

def normalize_azure_openai_endpoint(endpoint):
    """Convert an Azure Foundry /models endpoint to an Azure OpenAI endpoint."""
    endpoint = endpoint.rstrip("/")
    if ".services.ai.azure.com/models" in endpoint:
        resource_name = endpoint.split("//", 1)[1].split(".", 1)[0]
        endpoint = f"https://{resource_name}.openai.azure.com"
    if "/openai/deployments" in endpoint or endpoint.endswith("/models"):
        raise ValueError("Set AZURE_OPENAI_ENDPOINT to the Azure OpenAI resource endpoint, not an inference/deployment URL.")
    return endpoint.rstrip("/") + "/"

required_env = [
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_API_VERSION",
    "AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME",
]
missing_env = [name for name in required_env if not os.getenv(name) or os.getenv(name).startswith("replace-")]
if missing_env:
    raise ValueError(f"Missing Azure environment values: {', '.join(missing_env)}")

os.environ["AZURE_OPENAI_ENDPOINT"] = normalize_azure_openai_endpoint(
    os.environ["AZURE_OPENAI_ENDPOINT"]
)


C:\Users\bsmun\AppData\Local\Temp\ipykernel_25448\2798604868.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.docstore.document import Document


### Define document path

In [4]:
path = r"F:\genai-langchain-tutorials\pdf\Understanding_Climate_Change.pdf"

### Encode the pdf to vector store and return split document from the step before to create BM25 instance

In [5]:
def encode_pdf_and_get_split_documents(path, chunk_size=1000, chunk_overlap=200):
    """
    Encodes a PDF book into a vector store using OpenAI embeddings.

    Args:
        path: The path to the PDF file.
        chunk_size: The desired size of each text chunk.
        chunk_overlap: The amount of overlap between consecutive chunks.

    Returns:
        A tuple of (FAISS vector store, cleaned text documents).
    """
    from langchain_community.document_loaders import PyPDFLoader
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_openai import AzureOpenAIEmbeddings
    from langchain_community.vectorstores import FAISS
    
    # Load PDF documents
    loader = PyPDFLoader(path)
    documents = loader.load()

    # Split documents into chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len
    )
    texts = text_splitter.split_documents(documents)
    cleaned_texts = replace_t_with_space(texts)

    # Create embeddings and vector store
    endpoint = normalize_azure_openai_endpoint(os.environ["AZURE_OPENAI_ENDPOINT"])
    embeddings = AzureOpenAIEmbeddings(
        azure_deployment=os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME"],
        azure_endpoint=endpoint,
        api_key=os.environ["AZURE_OPENAI_API_KEY"],
        api_version=os.environ["AZURE_OPENAI_API_VERSION"],
    )
    vectorstore = FAISS.from_documents(cleaned_texts, embeddings)

    return vectorstore, cleaned_texts

### Create vectorstore and get the chunked documents

In [12]:
embeddings = AzureOpenAIEmbeddings(
    model="text-embedding-3-small",
    azure_deployment="text-embedding-3-small",
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version="2024-10-21"
)

In [9]:
vectorstore, cleaned_texts = encode_pdf_and_get_split_documents(path)

### Create a bm25 index for retrieving documents by keywords

In [14]:
from rank_bm25 import BM25Okapi

# Initalizing
corpus = [
    "Hello there good man!",
    "It is quite windy in London",
    "How is the weather today?"
]

tokenized_corpus = [doc.split(" ") for doc in corpus]

bm25 = BM25Okapi(tokenized_corpus)

# Ranking of documents
query = "windy London"
tokenized_query = query.split(" ")

doc_scores = bm25.get_scores(tokenized_query)
print(doc_scores)

doc_top_n = bm25.get_top_n(tokenized_query, corpus, n=1)
print(doc_top_n)

[0.         0.93729472 0.        ]
['It is quite windy in London']


In [21]:
def create_bm25_index(documents: List[Document]) -> BM25Okapi:
    """
    Create a BM25 index from the given documents.

    BM25 (Best Matching 25) is a ranking function used in information retrieval.
    It's based on the probabilistic retrieval framework and is an improvement over TF-IDF.

    Args:
    documents (List[Document]): List of documents to index.

    Returns:
    BM25Okapi: An index that can be used for BM25 scoring.
    """
    # Tokenize each document by splitting on whitespace
    # This is a simple approach and could be improved with more sophisticated tokenization
    tokenized_docs = [doc.page_content.split() for doc in documents]
    return BM25Okapi(tokenized_docs)

In [22]:
bm25 = create_bm25_index(cleaned_texts)

In [32]:
import numpy as np

# First chunk's text
chunk = cleaned_texts[0].page_content

# First chunk's BM25 term frequencies
doc = bm25.doc_freqs[0]

# Vocabulary and BM25 representation
words = list(bm25.idf.keys())

bm25_vector = np.array([
    doc.get(word, 0) * bm25.idf[word]
    for word in words
])

print("FIRST CHUNK")
print("=" * 60)
print(chunk[:1000])

print("\nBM25 VECTOR")
print("=" * 60)
print("Vector shape:", bm25_vector.shape)
print("Vector values:")
print(bm25_vector)

FIRST CHUNK
Understanding Climate Change 
Chapter 1: Introduction to Climate Change 
Climate change refers to significant, long-term changes in the global climate. The term 
"global climate" encompasses the planet's overall weather patterns, including temperature, 
precipitation, and wind patterns, over an extended period. Over the past century, human 
activities, particularly the burning of fossil fuels and deforestation, have significantly 
contributed to climate change. 
Historical Context 
The Earth's climate has changed throughout history. Over the past 650,000 years, there have 
been seven cycles of glacial advance and retreat, with the abrupt end of the last ice age about 
11,700 years ago marking the beginning of the modern climate era and human civilization. 
Most of these climate changes are attributed to very small variations in Earth's orbit that 
change the amount of solar energy our planet receives. During the Holocene epoch, which

BM25 VECTOR
Vector shape: (2630,)
Vecto

In [33]:
for word, value in zip(words, bm25_vector):
    if value != 0:
        print(f"{word:25s} {value:.4f}")

Understanding             3.6428
Climate                   0.5526
Change                    2.4212
Chapter                   1.0449
1:                        4.1641
Introduction              4.1641
to                        3.6285
change                    1.6849
refers                    3.6428
significant,              4.1641
long-term                 2.4904
changes                   5.2891
in                        1.8142
the                       9.9784
global                    1.1538
climate.                  3.6428
The                       1.9851
term                      4.1641
"global                   4.1641
climate"                  4.1641
encompasses               3.6428
planet's                  4.1641
overall                   3.6428
weather                   2.6445
patterns,                 6.0678
including                 2.0178
temperature,              3.6428
precipitation,            4.1641
and                       3.6285
wind                      2.1203
over      

In [34]:
doc = bm25.doc_freqs[0]

for index, word in enumerate(words):
    if doc.get(word, 0) != 0:
        print(
            f"Index: {index:4} | "
            f"Word: {word:20} | "
            f"Weight: {bm25_vector[index]:.4f}"
        )

Index:    0 | Word: Understanding        | Weight: 3.6428
Index:    1 | Word: Climate              | Weight: 0.5526
Index:    2 | Word: Change               | Weight: 2.4212
Index:    3 | Word: Chapter              | Weight: 1.0449
Index:    4 | Word: 1:                   | Weight: 4.1641
Index:    5 | Word: Introduction         | Weight: 4.1641
Index:    6 | Word: to                   | Weight: 3.6285
Index:    7 | Word: change               | Weight: 1.6849
Index:    8 | Word: refers               | Weight: 3.6428
Index:    9 | Word: significant,         | Weight: 4.1641
Index:   10 | Word: long-term            | Weight: 2.4904
Index:   11 | Word: changes              | Weight: 5.2891
Index:   12 | Word: in                   | Weight: 1.8142
Index:   13 | Word: the                  | Weight: 9.9784
Index:   14 | Word: global               | Weight: 1.1538
Index:   15 | Word: climate.             | Weight: 3.6428
Index:   16 | Word: The                  | Weight: 1.9851
Index:   17 | 